# ResNet50 Transfer-Learning

In [2]:
import os
# PROXY für Bosch-Netzwerk EINRICHTEN (bevor torchvision/requests importiert werden!)
os.environ['HTTP_PROXY'] = 'http://rb-proxy-de.bosch.com:8080'
os.environ['HTTPS_PROXY'] = 'http://rb-proxy-de.bosch.com:8080'

import time
import copy
import zipfile
import urllib.request

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, models, transforms

########################
# 1. ECHTE DATEN AUTOMATISCH DOWNLOADEN
########################

data_dir = "data/hymenoptera_data"
zip_path = f"{data_dir}.zip"

# Erstelle Ordner
os.makedirs("data", exist_ok=True)

# Download + Entpacken (offizieller PyTorch-Datensatz: 244 Train, 153 Val Bilder) [web:28][web:31]
print("📥 Lade Hymenoptera-Datensatz (Ameisen vs. Bienen) herunter...")
if not os.path.exists(data_dir):
    print("↓ Download von https://download.pytorch.org/tutorial/hymenoptera_data.zip")
    urllib.request.urlretrieve(
        "https://download.pytorch.org/tutorial/hymenoptera_data.zip",
        zip_path
    )
    print("✅ ZIP heruntergeladen!")
    
    print("📦 Entpacke...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data")
    print("✅ Entpackt! Datensatz bereit.")
    
    # ZIP löschen (optional)
    os.remove(zip_path)
else:
    print("✅ Datensatz bereits vorhanden!")

########################
# 2. Einstellungen
########################

num_classes = 2  # ants, bees
batch_size = 8
num_epochs = 10
lr = 0.001
step_size = 7
gamma = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

########################
# 3. DataLoader
########################

data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    "val": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

image_datasets = {
    x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
    for x in ["train", "val"]
}

dataloaders = {
    x: torch.utils.data.DataLoader(
        image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=4
    )
    for x in ["train", "val"]
}

dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val"]}
class_names = image_datasets["train"].classes

print("Train size:", dataset_sizes["train"])
print("Val size  :", dataset_sizes["val"])
print("Klassen   :", class_names)

########################
# 4. ResNet50 + Transfer Learning
########################

model = models.resnet50(weights="IMAGENET1K_V1")

# Alle Parameter trainierbar (Fine-Tuning)
for param in model.parameters():
    param.requires_grad = True

# FC-Schicht ersetzen für 2 Klassen
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

########################
# 5. Loss, Opti, Scheduler
########################

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
scheduler = lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

########################
# 6. Training
########################

def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == "train":
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s")
    print(f"Best val Acc: {best_acc:4f}")

    model.load_state_dict(best_model_wts)
    return model

model = train_model(model, criterion, optimizer, scheduler, num_epochs)

########################
# 7. BEISPIEL-AUSGABE
########################

model.eval()
with torch.no_grad():
    inputs, labels = next(iter(dataloaders["val"]))
    inputs, labels = inputs.to(device), labels.to(device)
    outputs = model(inputs)
    _, preds = torch.max(outputs, 1)

print("\nBeispiellabels (GT):", [class_names[l] for l in labels[:8].cpu()])
print("Vorhersagen       :", [class_names[p] for p in preds[:8].cpu()])


📥 Lade Hymenoptera-Datensatz (Ameisen vs. Bienen) herunter...
↓ Download von https://download.pytorch.org/tutorial/hymenoptera_data.zip
✅ ZIP heruntergeladen!
📦 Entpacke...
✅ Entpackt! Datensatz bereit.
Device: cuda
Train size: 244
Val size  : 153
Klassen   : ['ants', 'bees']


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\wug2si/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:09<00:00, 11.3MB/s]


Epoch 1/10
------------------------------
train Loss: 0.5309 Acc: 0.6926
val Loss: 0.2770 Acc: 0.8889

Epoch 2/10
------------------------------
train Loss: 0.3412 Acc: 0.8484
val Loss: 0.1686 Acc: 0.9412

Epoch 3/10
------------------------------
train Loss: 0.2693 Acc: 0.8811
val Loss: 0.1416 Acc: 0.9542

Epoch 4/10
------------------------------
train Loss: 0.2002 Acc: 0.9180
val Loss: 0.2234 Acc: 0.9216

Epoch 5/10
------------------------------
train Loss: 0.1813 Acc: 0.9180
val Loss: 0.1810 Acc: 0.9216

Epoch 6/10
------------------------------
train Loss: 0.1760 Acc: 0.9221
val Loss: 0.3147 Acc: 0.8954

Epoch 7/10
------------------------------
train Loss: 0.1670 Acc: 0.9426
val Loss: 0.2899 Acc: 0.9085

Epoch 8/10
------------------------------
train Loss: 0.0731 Acc: 0.9836
val Loss: 0.2114 Acc: 0.9542

Epoch 9/10
------------------------------
train Loss: 0.1176 Acc: 0.9467
val Loss: 0.1962 Acc: 0.9542

Epoch 10/10
------------------------------
train Loss: 0.0971 Acc: 0.9590